# Faruq-v3 — ontology error attribution

Inference validation-only untuk memisahkan error sibling, same-family, dan cross-family pada D0/C0/S0. Tidak ada training atau akses test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-ontology-marginal-v1/C0_seed42/weights/best.pt',
    'experiments/faruq-v3-ontology-marginal-v1/S0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
C0 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-ontology-marginal-v1/C0_seed42/weights/best.pt')
S0 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-ontology-marginal-v1/S0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-ontology-marginal-v1/error_attribution'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', OUTPUT_ROOT)

In [ ]:
from coffee_detector.analysis.ontology_error_attribution import run_ontology_error_attribution
result = run_ontology_error_attribution(
    DATA_ROOT, D0, C0, S0, OUTPUT_ROOT, device='0'
)
assert result['training_executed'] is False
assert result['test_images_accessed'] is False
print('ATTRIBUTION SELESAI')

In [ ]:
import pandas as pd
from IPython.display import display
rows = []
for code, report in result['models'].items():
    rows.append({'model': code, 'wrong_class': report['wrong_class'], **report['category_counts']})
display(pd.DataFrame(rows))
display(pd.DataFrame(result['S0_vs_D0']['largest_pair_increases']))
print('S0-D0 CATEGORY DELTA:', result['S0_vs_D0']['category_count_deltas'])
print('INTERPRETATION:', result['interpretation'])
print('NEXT:', result['next_action'])
print('SUMMARY:', result['summary'])
print('Kirim dua tabel dan interpretation. Jangan training.')